In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
stocks = [
    "AAPL",  # Apple
    "GOOGL", # Google
    "META",  # Meta
    "AMZN",  # Amazon
    "MSFT",  # Microsoft
    "NVDA",  # Nvidia
    "JPM",   # JP Morgan
    "GS",    # Goldman Sachs
    "TSLA",  # Tesla
    "RIVN",  # Rivian
]
#get end and start date and convert to string
#Airflow will stream one year old data till now!
END_DATE = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today()-timedelta(days=365)).strftime('%Y-%m-%d')
print(f" FROM: {START_DATE}")
print(f" TO: {END_DATE}")


In [0]:
all_stocks_data = []

for ticker in stocks:
    curr_stock = yf.download(ticker, start=START_DATE, end=END_DATE,auto_adjust=True)

    curr_stock.columns = [col[0] for col in curr_stock]
    curr_stock = curr_stock.reset_index()
    curr_stock['ticker'] = ticker
    curr_stock['ingestion_time']=datetime.today()
    all_stocks_data.append(curr_stock)

final_df = pd.concat(all_stocks_data,ignore_index=True)

final_df = final_df.rename(columns={
    'Date'  : 'trade_date',
    'Close' : 'close_price',
    'High'  : 'high_price',
    'Low'   : 'low_price',
    'Open'  : 'open_price',
    'Volume': 'volume'
})

display(final_df)


In [0]:
spark_df = spark.createDataFrame(final_df)
spark_df.printSchema()

In [0]:
display(spark.sql("SHOW CATALOGS"))

## CREATING DELTA TABLES FOR FURTHER USGAE

In [0]:
spark.sql("DROP SCHEMA IF EXISTS raj.stock_market CASCADE")

In [0]:
spark.sql("DROP SCHEMA IF EXISTS raj.bronze_olist CASCADE")
spark.sql("DROP SCHEMA IF EXISTS raj.gold_olist CASCADE")
spark.sql("DROP SCHEMA IF EXISTS raj.silver_olist CASCADE")
spark.sql("DROP SCHEMA IF EXISTS raj.default CASCADE")


In [0]:
display(spark.sql("SHOW SCHEMAS IN raj"))

In [0]:
spark_df.write.format('delta').mode("append").option("mergeSchema", "true").saveAsTable("raj.bronze.stock_prices")